In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import glob
import numpy as np
import os
import gzip
import glob
# from bs4 import BeautifulSoup
import decoupler as dc

plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
import scanpy as sc
# import scanpy.external as sce
# import loompy as lp
import anndata as ad
# from scipy.io import mmread
# import re
# from scipy.sparse import coo_matrix, csr_matrix
# from scipy.spatial.distance import cosine
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import gc

sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
# from MulticoreTSNE import MulticoreTSNE as TSNE

Gene Expression Signature Analysis Process from paper:

Data Processing:

Normalization: The gene expression microarray data normalized using the Robust Multi-array Average (RMA) method, followed by quantile normalization- R package version 2.15.1.

Analysis and Generation of Gene Expression Signatures:
Hierarchical Clustering: Hierarchical clustering was performed on the gene signatures derived from the 1,000 most highly differentially expressed genes in hematopoietic stem cells (HSCs) and granulocyte-macrophage (GM) lineage-committed cells. 
This clustering divided the hematopoietic hierarchy into three main groups: highly immature HSCs, multipotent progenitors (MPP2, MPP3, MPP4), and GM lineage-committed cells.

Principal Component Analysis (PCA): PCA was conducted to identify the main drivers of gene expression variation. Different principal components (PCs) separated the GM-committed myeloid differentiation axis, HSC subtypes, and MPP subsets based on specific gene expression patterns.

Gene Ontology (GO) Analysis: GO analysis was performed on gene signatures representing the 1,000 most highly expressed genes in each MPP subset. This analysis identified biological processes and pathways enriched in each subset, highlighting their unique molecular features.

OUTLINE FOR ANNOTATION:

Passegue- GEO datasets- each celltype population find most unique differentially expressed genes against other populations, fdr < 0.01 (look across all files within cell type)- top 100 most specific genes to that population- exclusive lists of genes for each
vector: expression for 100 genes


score each cluster with each cell population from Passegue data- the top 100 identified genes to cross-correlate between the clusters (compare the expression of the genes (NOT abosolute values- whether they are differentiall expressed in that cluster: I ran sc.tl.rank_genes_groups_8(adata, 'leiden', method='wilcoxon') to create this adata so they are ranked within clusters- need to access)). For each cluster, highest score (most correlation) gets assigned as label to that cluster

plot umap with labels and see


## RUN only to regenerate seurat object

In [ ]:
# /home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/raw_10x_data/Pool140_1/outs/filtered_feature_bc_matrix

# example /home/rav589/scratch/SC-RNASEQ/geo_submission_scRNA-seq/Pool140_1/Pool140_1_barcodes.tsv.gz, etc.
#Pool140_1 to 13
filenames = glob.glob('/home/rav589/scratch/SC-RNASEQ/geo_submission_scRNA-seq/Pool140_*')

In [ ]:
adatas = [sc.read_10x_mtx(filename, gex_only = False, prefix=f'Pool140') for filename in filenames]

In [ ]:
# Get list of matching directories
filenames = glob.glob('/home/rav589/scratch/SC-RNASEQ/geo_submission_scRNA-seq/Pool140_*')

# Sort filenames to ensure correct order
filenames.sort()

# Extract pool numbers dynamically and read 10x matrices
adatas = [
    sc.read_10x_mtx(filename, gex_only=False, prefix=f"Pool140_{os.path.basename(filename).split('_')[-1]}_")
    for filename in filenames
]

In [ ]:
#integration step

#original: simple concatenation
adata = adatas[0].concatenate(adatas[1:])

# new version: with harmony

# adata.var_names_make_unique()
adata

In [ ]:
# adata.write('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_adata_unfiltered.h5ad')
adata = sc.read_h5ad('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_adata_unfiltered.h5ad')
adata

In [ ]:
adata = adata[adata.obs['batch'] != '0'].copy()

In [ ]:
# # uncomment to use harmony
# sc.pp.pca(adata)
# sce.pp.harmony_integrate(adata, 'batch')
# adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
# adata

In [ ]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("mt-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains("^Hb")

In [ ]:
adata.var["mt"].value_counts()

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
adata

In [ ]:
p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
# sc.pl.violin(adata, 'total_counts')
p2 = sc.pl.violin(adata, "pct_counts_mt")
p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
adata.obs["pct_counts_mt"].hist()

In [ ]:
from scipy import stats

def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * stats.median_abs_deviation(M)) | (
        np.median(M) + nmads * stats.median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
adata.obs["outlier"] = (
    is_outlier(adata, "log1p_total_counts", 5)
    | is_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
    | (adata.obs["total_counts"] < 1000)
    | (adata.obs["n_genes_by_counts"] < 500)
)

adata.obs.outlier.value_counts()

#which samples

In [ ]:
adata.obs[adata.obs['outlier'] == True]['batch'].value_counts()

In [ ]:
adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 3) | (
    adata.obs["pct_counts_mt"] > 10
)
adata.obs.mt_outlier.value_counts()

In [ ]:
print(f"Total number of cells: {adata.n_obs}")
adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()

print(f"Number of cells after filtering of low quality cells: {adata.n_obs}")

In [ ]:
p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
# quality control: filter_cells, filter_genes, and filter_highly_variable_genes
print(f"Total number of genes: {adata.n_vars}")

adata.layers["counts"] = adata.X.copy()
sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_genes(adata, min_cells=3)

print(f"Number of genes after cell filter: {adata.n_vars}")

In [ ]:
adata

In [ ]:
sc.pp.scrublet(adata, batch_key="batch")
adata

In [ ]:
adata.obs['doublet_score'].describe()

In [ ]:
(adata.obs['doublet_score'] > 0.01).value_counts()
# adata.obs['predicted_doublet'].value_counts()

In [ ]:
adata = adata[adata.obs['doublet_score'] > 0.01].copy()
adata

In [ ]:
adata.write('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_adata_quality_filtered_strict.h5ad')

In [ ]:
adata = sc.read_h5ad('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_adata_quality_filtered_strict.h5ad')
adata

#### ambient rna

In [ ]:
import anndata2ri
import logging

import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
%%R
library(SoupX)

In [ ]:
adata_pp = adata.copy()
sc.pp.normalize_per_cell(adata_pp)
sc.pp.log1p(adata_pp)

In [ ]:
sc.pp.pca(adata_pp)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="soupx_groups")

# Preprocess variables for SoupX
soupx_groups = adata_pp.obs["soupx_groups"]

In [ ]:
del adata_pp

In [ ]:
cells = adata.obs_names
genes = adata.var_names
data = adata.X.T

In [ ]:
#FIX!: Our raw data
adata_raw = sc.read_10x_h5(
    filename="raw_feature_bc_matrix.h5",
    backup_url="https://figshare.com/ndownloader/files/39546217",
)
adata_raw.var_names_make_unique()
data_tod = adata_raw.X.T

In [ ]:
del adata_raw

In [ ]:
%%R -i data -i data_tod -i genes -i cells -i soupx_groups -o out 

# specify row and column names of data
rownames(data) = genes
colnames(data) = cells
# ensure correct sparse format for table of counts and table of droplets
data <- as(data, "sparseMatrix")
data_tod <- as(data_tod, "sparseMatrix")

# Generate SoupChannel Object for SoupX 
sc = SoupChannel(data_tod, data, calcSoupProfile = FALSE)

# Add extra meta data to the SoupChannel object
soupProf = data.frame(row.names = rownames(data), est = rowSums(data)/sum(data), counts = rowSums(data))
sc = setSoupProfile(sc, soupProf)
# Set cluster information in SoupChannel
sc = setClusters(sc, soupx_groups)

# Estimate contamination fraction
sc  = autoEstCont(sc, doPlot=FALSE)
# Infer corrected table of counts and rount to integer
out = adjustCounts(sc, roundToInt = TRUE)

In [ ]:
adata.layers["counts"] = adata.X
adata.layers["soupX_counts"] = out.T
adata.X = adata.layers["soupX_counts"]

#### rna and protein split

In [ ]:
protein = adata[:, adata.var["feature_types"] == "Antibody Capture"].copy()
rna = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()

In [ ]:
rna.shape

In [ ]:
rna

In [ ]:
# # Storing the counts for later use
# rna.layers["counts"] = rna.X.copy()

# Normalize total: QC
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

In [ ]:
# Finding highly variable genes using count data
sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pp.pca(rna)
sc.pp.neighbors(rna)
sc.tl.umap(rna)

In [ ]:
s_genes = [
    'Mcm5', 'Pcna', 'Tyms', 'Fen1', 'Mcm2', 'Mcm4', 'Rrm1', 'Ung',
    'Gins2', 'Mcm6', 'Cdca7', 'Dtl', 'Prim1', 'Uhrf1', 'Cenpu', 'Hells',
    'Rfc2', 'Nasp', 'Rad51ap1', 'Gmnn', 'Wdr76', 'Slbp', 'Ccne2', 'Ubr7',
    'Pold3', 'Msh2', 'Atad2', 'Rad51', 'Rpa2', 'Cdc45', 'Cdc6', 'Exo1',
    'Tipin', 'Dscc1', 'Blm', 'Casp8ap2', 'Usp1', 'Clspn', 'Pola1', 'Chaf1b',
    'Brip1', 'E2f8'
]

g2m_genes = [
    'Hmgb2', 'Cdk1', 'Nusap1', 'Ube2c', 'Birc5', 'Tpx2', 'Top2a', 'Ndc80',
    'Ckap2l', 'Ckap2', 'Aurkb', 'Bub1', 'Kif11', 'Anp32e', 'Tubb4b', 'Gtse1',
    'Kif20b', 'Hjurp', 'Cdca3', 'Cdc20', 'Ttk', 'Cdc25c', 'Kif2c', 'Rangap1',
    'Ncapd2', 'Dlgap5', 'Cdca2', 'Cdca8', 'Ect2', 'Kif23', 'Hmmr', 'Aurka',
    'Psrc1', 'Anln', 'Lbr', 'Ckap5', 'Cenpe', 'Ctcf', 'Nek2', 'G2e3', 'Gas2l3',
    'Cbx5', 'Cenpa'
]


# Score the genes for cell cycle effects
sc.tl.score_genes_cell_cycle(rna, s_genes=s_genes, g2m_genes=g2m_genes)

# Plot the results to check for cell cycle effects
sc.pl.violin(rna, ['S_score', 'G2M_score'], groupby='leiden_8', jitter=0.4)
sc.pl.scatter(rna, 'S_score', 'G2M_score', color='leiden_8')

# Optionally, if you want to visualize the scores on UMAP or t-SNE
sc.tl.umap(rna)
sc.pl.umap(rna, color=['S_score', 'G2M_score'])

In [ ]:
rna_cc = rna.copy()

In [ ]:
sc.pp.regress_out(rna_cc, ['S_score', 'G2M_score'])
sc.pp.scale(rna_cc)

In [ ]:
sc.tl.leiden(rna_cc, resolution=0.8, key_added="leiden_8")

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna_cc, color='leiden_8', size=1, show=False)
plt.show()

In [ ]:
# sc.tl.leiden(rna, resolution=0.5, key_added="leiden_5")
# sc.tl.leiden(rna, resolution=0.6, key_added="leiden_6")
# sc.tl.leiden(rna, resolution=0.7, key_added="leiden_7")
sc.tl.leiden(rna, resolution=0.8, key_added="leiden_8")
# sc.tl.leiden(rna, resolution=0.9, key_added="leiden_9")

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna, color='leiden_8', size=1, show=False)
plt.savefig('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/umap_custers_leiden_8_qc.pdf', format='pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna, color='leiden_6', size=1, show=False)
plt.savefig('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/umap_custers_leiden_6_qc.pdf', format='pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna, color='leiden_7', size=1, show=False)
plt.savefig('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/umap_custers_leiden_7_qc.pdf', format='pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna, color='leiden_8', size=1, show=False)
plt.savefig('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/umap_custers_leiden_8_qc.pdf', format='pdf')

plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna, color='leiden_9', size=1, show=False)
plt.show()

In [ ]:
# sc.tl.rank_genes_groups(rna, 'leiden_5', method='wilcoxon', key_added='rank_genes_groups_5')
# sc.tl.rank_genes_groups(rna, 'leiden_6', method='wilcoxon', key_added='rank_genes_groups_6')
# sc.tl.rank_genes_groups(rna, 'leiden_7', method='wilcoxon', key_added='rank_genes_groups_7')
sc.tl.rank_genes_groups(rna, 'leiden_8', method='wilcoxon', key_added='rank_genes_groups_8')
# sc.tl.rank_genes_groups(rna, 'leiden_9', method='wilcoxon', key_added='rank_genes_groups_9')

In [ ]:
rna.write('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_filtered_rna_counts_qc_added_strict.h5ad')

In [ ]:
rna = sc.read_h5ad('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_res0.8_annotated_rna_qc_added_strict.h5ad')
# rna = sc.read_h5ad('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_filtered_rna_counts_qc_added_strict.h5ad')

In [ ]:
rna

In [ ]:
rna.obs['batch'].value_counts()

In [ ]:
protein

In [ ]:
sc.pp.normalize_total(protein, target_sum=1e4)
sc.pp.log1p(protein)

In [ ]:
sc.pp.pca(protein)
sc.pp.neighbors(protein) 

In [ ]:
sc.tl.leiden(protein, resolution= 0.8, key_added="protein_leiden")

In [ ]:
protein.obsp["protein_connectivities"] = protein.obsp["connectivities"].copy()
sc.tl.umap(protein)
sc.pl.umap(protein, color="protein_leiden", size=10)

In [ ]:
# protein.write('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_filtered_protein_counts_qc_added_strict.h5ad')

protein = sc.read_h5ad('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_filtered_protein_counts_qc_added_strict.h5ad')

In [ ]:
rna.obsm["protein"] = protein.to_df()
rna.obsm["protein_umap"] = protein.obsm["X_umap"]
rna.obs["protein_leiden"] = protein.obs["protein_leiden"]

rna.obsp["rna_connectivities"] = rna.obsp["connectivities"].copy()
rna.obsp["protein_connectivities"] = protein.obsp["protein_connectivities"]

In [ ]:
sc.pl.umap(rna, color=["leiden_8", "protein_leiden"], size=10, show=False)  # Disable show to delay plot rendering
plt.savefig("/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/umap_leiden_8_protein_leiden.pdf", format='pdf', bbox_inches='tight')
plt.close()  # Close the plot to free up memory

# Generate the UMAP plot for protein data
sc.pl.embedding(rna, basis="protein_umap", color=["leiden_8", "protein_leiden"], size=10, show=False)  # Disable show to delay plot rendering
plt.savefig("/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/protein_umap_leiden_8_protein_leiden.pdf", format='pdf', bbox_inches='tight')
plt.close() 


In [ ]:
protein.var['gene_ids']

### Trying joint clustering approach: DO NOT RUN

In [ ]:
def join_graphs_max(g1: "sparse.spmatrix", g2: "sparse.spmatrix"):
    """Take the maximum edge value from each graph."""
    out = g1.copy()
    mask = g1 < g2
    out[mask] = g2[mask]

    return out

In [ ]:
rna.obsp["connectivities"] = join_graphs_max(rna.obsp["rna_connectivities"], rna.obsp["protein_connectivities"])
sc.tl.leiden(rna, key_added="joint_leiden")

In [ ]:
sc.pl.umap(rna, color="joint_leiden", size=5)

In [ ]:
sns.heatmap(pd.crosstab(rna.obs["joint_leiden"], rna.obs["protein_leiden"], normalize="index"))

In [ ]:
sns.heatmap(pd.crosstab(rna.obs["joint_leiden"], rna.obs["leiden"], normalize="index"))

### CITE-SEQ: Protein markers analysis

In [ ]:
protein.obs['leiden'] = rna.obs['leiden_8']
protein.obs['annotation'] = rna.obs['cell_type']

In [ ]:
protein_markers = protein.var.index.tolist()

# Generate the dotplot for protein data
sc.pl.dotplot(
    protein,
    var_names=protein_markers,
    groupby='annotation',
    figsize=(6, 6),
    show=False  # Disable show to delay plot rendering
)

# Save the plot as a PDF
plt.savefig('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/dotplot_protein_markers_celltype.pdf', format='pdf', bbox_inches='tight')
plt.close()  # Close the plot to free up memory


In [ ]:
# CORRELATION plot cite-seq markers data from protein, same cite-seq markers rna expression

protein_data = protein[:, protein.var_names].X.toarray()
rna_genes = ['Flt3', 'Cd48', 'Itga2b', 'Itgb3', 'Cd34', 'Slamf1', 'Cd9', 'Il7r']
rna_data = rna[:, rna_genes].X.toarray()

protein_df = pd.DataFrame(protein_data, index=protein.obs_names, columns=protein.var_names)
rna_df = pd.DataFrame(rna_data, index=rna.obs_names, columns=protein.var_names)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42).fit(rna_df)

rna_df['Cluster'] = kmeans.labels_
rna_df_sorted = rna_df.sort_values('Cluster').drop(columns='Cluster')

plt.figure(figsize=(10, 8))
ax = sns.heatmap(rna_df_sorted, cmap="viridis", cbar_kws={'label': 'Expression Level'})
plt.title('Heatmap of RNA Expression for CITE-seq Markers')
plt.xlabel('Markers')
plt.ylabel('Cells')
ax.set_yticklabels([])
plt.grid(False)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42).fit(protein_df)

protein_df['Cluster'] = kmeans.labels_
protein_df_sorted = protein_df.sort_values('Cluster').drop(columns='Cluster')

plt.figure(figsize=(10, 8))
ax = sns.heatmap(protein_df_sorted, cmap="viridis", cbar_kws={'label': 'Expression Level'})
plt.title('Heatmap of CITE-seq')
plt.xlabel('Markers')
plt.ylabel('Cells')
ax.set_yticklabels([])
plt.grid(False)
plt.show()

In [ ]:
protein_df.drop(columns=['Cluster'], inplace=True)
correlation_matrix = pd.DataFrame(index=protein_df.columns, columns=protein_df.columns)

for protein_marker in protein_df.columns:
    for protein_marker_2 in protein_df.columns:
        correlation_matrix.loc[protein_marker, protein_marker_2] = protein_df[protein_marker].corr(protein_df[protein_marker_2])

correlation_matrix = correlation_matrix.astype(float)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=False, cmap="vlag", cbar_kws={'label': 'Correlation'})
plt.title('CITE-Seq')
plt.grid(False)
plt.show()

In [ ]:
rna_df.drop(columns=['Cluster'], inplace=True)
correlation_matrix = pd.DataFrame(index=rna_df.columns, columns=rna_df.columns)

for protein_marker in rna_df.columns:
    for protein_marker_2 in rna_df.columns:
        correlation_matrix.loc[protein_marker, protein_marker_2] = rna_df[protein_marker].corr(rna_df[protein_marker_2])

correlation_matrix = correlation_matrix.astype(float)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=False, cmap="vlag", cbar_kws={'label': 'Correlation'})
plt.title('RNA-Seq')
plt.grid(False)
plt.show()

In [ ]:
correlation_matrix = pd.DataFrame(index=protein_df.columns, columns=rna_df.columns)

for protein_marker in protein_df.columns:
    for rna_marker in rna_df.columns:
        correlation_matrix.loc[protein_marker, rna_marker] = protein_df[protein_marker].corr(rna_df[rna_marker])

correlation_matrix = correlation_matrix.astype(float)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=False, cmap="vlag", cbar_kws={'label': 'Correlation'})
plt.title('CITE-seq - RNA Expression')
plt.xlabel('CITE-seq')
plt.ylabel('RNA Expression')
plt.grid(False)
plt.show()

In [ ]:
plotdf = sc.get.obs_df(
    rna,
    obsm_keys=[("X_umap", i) for i in range(2)]
)

# Add the protein expression data to plotdf
for gene in protein.var_names:
    plotdf[gene] = protein_df[gene]

def embedding_chart(df: pd.DataFrame, coord_pat: str, size=5):
    """
    Plot embedding with scatter plot for cells, colored by protein expression or any other feature.
    df: DataFrame with UMAP coordinates and expression data.
    coord_pat: String pattern to match coordinate columns (e.g., 'umap').
    size: Point size for the scatter plot.
    """
    # Get UMAP or other embedding coordinates
    x, y = df.columns[df.columns.str.contains(coord_pat)]

    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x=df[x], 
        y=df[y], 
        s=size, 
        color='grey',  # Default color, unless overridden
        alpha=0.6
    )
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f'Embedding: {coord_pat}')
    plt.show()

def plot_protein_expression(df: pd.DataFrame, coord_pat: str, protein_col: str, size=5, cmap="coolwarm", save_path=None):
    """
    Scatter plot with embedding coordinates, colored by protein expression levels.
    df: DataFrame with UMAP coordinates and expression data.
    coord_pat: String pattern to match coordinate columns (e.g., 'umap').
    protein_col: The protein expression column to be visualized.
    size: Point size for the scatter plot.
    cmap: Colormap for expression levels.
    save_path: File path to save the plot as a PDF.
    """
    # Get UMAP or other embedding coordinates
    x, y = df.columns[df.columns.str.contains(coord_pat)]

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        df[x], df[y], c=df[protein_col], s=size, cmap=cmap, alpha=0.7, rasterized=True
    )
    plt.colorbar(scatter, label=protein_col)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(False)
    plt.title(f'{protein_col}')

    if save_path:
        plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()

# Loop through all protein columns and generate UMAP plots
for protein in plotdf.columns[2:]:  # Assuming protein columns start from index 2
    save_path = f'/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/umap_{protein}_rasterized.pdf'
    plot_protein_expression(plotdf, "umap", protein_col=protein, size=2, save_path=save_path)


In [ ]:
plotdf.columns

### Using gene Markers from Collins/Passegue Paper
tutorial: https://decoupler-py.readthedocs.io/en/latest/notebooks/cell_annotation.html

In [ ]:
# markers = dc.get_resource('PanglaoDB')
# markers = pd.read_csv('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/raw_10x_data/feature_reference.csv', sep=',')

In [ ]:
df = pd.read_csv('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/Pietras_GSE68529/Collins_Table_S4.csv', sep=',')
df

In [ ]:
markers = pd.DataFrame(columns=['gene_symbol', 'cell_type'])

for column in df.columns:
    expanded_col = df[column].str.split(' ').explode()
    temp_df = pd.DataFrame({'gene_symbol': expanded_col, 'cell_type': column})
    print(temp_df)
    markers = pd.concat([markers, temp_df], ignore_index=True)

markers = markers.dropna(subset=['gene_symbol'])
markers

In [ ]:
markers['cell_type'].value_counts()

In [ ]:
rna

In [ ]:
dc.run_ora(
    mat=rna,
    net=markers,
    source='cell_type',
    target='gene_symbol',
    min_n=3,
    verbose=True,
    use_raw=False
)

In [ ]:
rna=rna_copy

In [ ]:
rna.uns['rank_genes_groups_8'].keys()

In [ ]:
#filtering to log2fc > 0 and doing differential expression analysis, then ora
ora_estimate = pd.DataFrame()

for i in range(20):
    degs_i = sc.get.rank_genes_groups_df(rna, group=str(i), log2fc_min=0, key='rank_genes_groups_8')
    degs_i.set_index('names', inplace=True)

    ora_i = dc.get_ora_df(degs_i, net=markers, source='cell_type', target='gene_symbol', verbose=True)
    ora_i['group'] = i
    ora_estimate = pd.concat([ora_estimate, ora_i], ignore_index=True)

ora_estimate

In [ ]:
ora_estimate[ora_estimate['group'] == 10].sort_values(by='Combined score', ascending=False)

In [ ]:
ora_estimate.to_csv('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/rna_ora_estimate.csv', index=False)

In [ ]:
# rna.obsm['ora_estimate'].columns

In [ ]:
acts = dc.get_acts(rna, obsm_key='ora_estimate')

# We need to remove inf and set them to the maximum value observed for pvals=0
acts_v = acts.X.ravel()
max_e = np.nanmax(acts_v[np.isfinite(acts_v)])
acts.X[~np.isfinite(acts.X)] = max_e

acts

In [ ]:
acts.var_names

In [ ]:
population = 'GP'
sc.pl.umap(acts, color=[population], cmap='RdBu_r', show=False)
plt.savefig(f'/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/density/umap_{population}.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df = dc.rank_sources_groups(acts, groupby='leiden_8', reference='rest', method='t-test_overestim_var')
df

In [ ]:
order = ['LT-HSC', 'ST-HSC', 'MPP2', 'MPP3', 'MPP4', 'MkP', 'BaP ', 'Pre-CFU-E ', 'CFU-E ', 'GMP_mult ', 'cMoP ', 'GP']
df['names'] = pd.Categorical(df['names'], categories=order, ordered=True)
group_order = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df

plt.figure(figsize=(12, 8))
bubble_plot = sns.scatterplot(
    data=df,
    x='names',
    y='group',
    size='statistic',
    hue='pvals_adj',
    hue_norm=(0,0.1),
    palette='RdBu_r',
    sizes=(50, 200),
    legend='brief'
)

plt.xlabel('Cell Types')
plt.ylabel('Clusters')
plt.xticks(rotation=90)
plt.legend()
bubble_plot.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

In [ ]:
n_ctypes = 3
# pval and statistic
ctypes_dict = df.groupby('group').head(n_ctypes).groupby('group')['names'].apply(lambda x: list(x)).to_dict()
ctypes_dict

annotations_df = pd.DataFrame()
annotations_df['collins_no_fdr'] = ctypes_dict.values()

annotations_df

In [ ]:
#FDR filtered 0.05
df = df[df['pvals_adj'] <= 0.05]
ctypes_dict = df.groupby('group').head(n_ctypes).groupby('group')['names'].apply(lambda x: list(x)).to_dict()
ctypes_dict

annotations_df['collins_fdr_0.05'] = ctypes_dict.values()
annotations_df

In [ ]:
# ora_estimate[ora_estimate['group'] == 6]

In [ ]:
ora_estimate_copy = ora_estimate.copy()
order = ['LT-HSC', 'ST-HSC', 'MPP2', 'MPP3', 'MPP4', 'MkP', 'BaP ', 'Pre-CFU-E ', 'CFU-E ', 'GMP_mult ', 'cMoP ', 'GP']

ora_estimate_copy['Term'] = pd.Categorical(ora_estimate_copy['Term'], categories=order, ordered=True)
group_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
ora_estimate_copy['group'] = pd.Categorical(ora_estimate_copy['group'], categories=group_order, ordered=True)
ora_estimate_copy

In [ ]:
# Create the bubble plot
plt.figure(figsize=(12, 8))
bubble_plot = sns.scatterplot(
    data=ora_estimate_copy,
    x='Term',
    y='group',
    size='Combined score',
    hue='FDR p-value',
    hue_norm=(0, 1),
    palette='RdBu_r',
    sizes=(100, 300),
    legend='brief'
)

# Customize plot labels and legend
plt.xlabel('Cell Types')
plt.ylabel('Clusters')
plt.xticks(rotation=90)
bubble_plot.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.yticks(ticks=range(len(group_order)), labels=group_order)
plt.show()

In [ ]:
n_ctypes = 3
# pval and statistic
ora_estimate_copy.sort_values('Combined score', ascending=False, inplace=True)
ctypes_dict = ora_estimate_copy.groupby('group').head(n_ctypes).groupby('group')['Term'].apply(lambda x: list(x)).to_dict()
ctypes_dict


annotations_df['collins_fc_0'] = ctypes_dict.values()
annotations_df

In [ ]:
ctypes_dict

In [ ]:
ora_estimate_copy = ora_estimate_copy[ora_estimate_copy['FDR p-value'] <= 0.05]
ora_estimate_copy.sort_values('Combined score', ascending=False, inplace=True)
ctypes_dict = ora_estimate_copy.groupby('group').head(n_ctypes).groupby('group')['Term'].apply(lambda x: list(x)).to_dict()
ctypes_dict


annotations_df['collins_fc_0_fdr_0.05'] = ctypes_dict.values()
annotations_df

## Pulling in Samples from Passegue's paper

In [ ]:
#gene level annotation 

with gzip.open('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/Pietras_GSE68529/GPL6246_gene_level.annot.gz', 'rt') as f:
    lines = f.readlines()

start_line = 0
for i, line in enumerate(lines):
    if line.startswith('!platform_table_begin'):
        start_line = i + 1
        break

with gzip.open('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/Pietras_GSE68529/GPL6246_gene_level.annot.gz', 'rt') as f:
    gene_metadata = pd.read_csv(f, sep='\t', skiprows=start_line)

In [ ]:
gene_metadata['ID'] = pd.to_numeric(gene_metadata['ID'], errors='coerce')
gene_metadata.sort_values(by='ID', inplace=True)
gene_metadata

In [ ]:
gene_metadata['Gene symbol']

In [ ]:
#get samples ID + gene expression values- normalized already

directory_path = '/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/Pietras_GSE68529'
html_files = glob.glob(os.path.join(directory_path, '*.html'))

dfs = {}

for file_path in html_files:
    with open(file_path, 'r') as file:
        html_content = file.read()

    soup = BeautifulSoup(html_content, 'html.parser')
    pre_tag_text = soup.find('pre').get_text()

    data = []
    for line in pre_tag_text.split('\n'):
        if line.strip() and not line.startswith('#') and not line.startswith('ID_REF'):
            id_ref, value = line.split()
            data.append({'ID_REF': id_ref, 'VALUE': value})

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    dfs[file_name] = pd.DataFrame(data)

dfs['GSM1674621_HSC_LT_1']

In [ ]:
#map gene info to each sample

# for df_name, df in dfs.items():
#     df['ID_REF'] = pd.to_numeric(df['ID_REF'], errors='coerce')
#     merged_df = pd.merge(df, gene_metadata, left_on='ID_REF', right_on='ID', how='inner')
#     merged_df.drop('ID', axis=1, inplace=True)
    
#     dfs[df_name] = merged_df

# dfs['GSM1674621_HSC_LT_1'].sort_values(by='ID_REF')

In [ ]:
# Combine data for each cell type into a single dataframe
cell_types = {
    'HSC_LT': ['GSM1674621_HSC_LT_1', 'GSM1674622_HSC_LT_2', 'GSM1674623_HSC_LT_3', 'GSM1674624_HSC_LT_4'],
    'HSC_ST': ['GSM1674625_HSC_ST_1', 'GSM1674626_HSC_ST_2', 'GSM1674627_HSC_ST_3'],
    'MPP_2': ['GSM1674628_MPP2_1', 'GSM1674629_MPP2_2', 'GSM1674630_MPP2_3'],
    'MPP_3': ['GSM1674631_MPP3_1', 'GSM1674632_MPP3_2', 'GSM1674633_MPP3_3'],
    'MPP_4': ['GSM1674634_MPP4_1', 'GSM1674635_MPP4_2', 'GSM1674636_MPP4_3', 'GSM1674637_MPP4_4', 'GSM1674638_MPP4_5'],
    'CMP': ['GSM1674639_CMP_1', 'GSM1674640_CMP_2', 'GSM1674641_CMP_3'],
    'GMP': ['GSM1674642_GMP_1', 'GSM1674643_GMP_2', 'GSM1674644_GMP_3', 'GSM1674645_GMP_4'],
    'PreGr': ['GSM1674646_PreGr_1', 'GSM1674647_PreGr_2', 'GSM1674648_PreGr_3'],
    'Gr': ['GSM1674649_Gr_1', 'GSM1674650_Gr_2', 'GSM1674651_Gr_3']
}

combined_dfs = {}
for cell_type, files in cell_types.items():
    combined_df_list = []
    for file in files:
        try:
            df = dfs[file].set_index('ID_REF')
            df = df[['VALUE']]
            df = df.rename(columns={'VALUE': f'VALUE_{file.split("_")[-1]}'})
            combined_df_list.append(df)
        except Exception as e:
            print(f"Error processing file {file}: {e}")
            continue

    try:
        combined_df = pd.concat(combined_df_list, axis=1)
        combined_df = combined_df.apply(pd.to_numeric, errors='coerce')  # Ensure all values are numeric
        combined_df["ID_REF"] = combined_df.index
        combined_dfs[cell_type] = combined_df
    except Exception as e:
        print(f"Error combining or averaging data for cell type {cell_type}: {e}")
        continue

In [ ]:
combined_dfs['HSC_LT']

In [ ]:
#make combined_df into adata object: cell types, values as gene expression values (replicates). goal: perform differential analysis (run code: sc.tl.rank_genes_groups_8(adata, 'cell_type', method='wilcoxon')) to get differentially expressed genes in each cell type. 

In [ ]:
all_data = []
for cell_type, df in combined_dfs.items():
    df['cell_type'] = cell_type  # Add cell type column
    df.reset_index(drop=True, inplace=True) 
    all_data.append(df)

print(all_data)
combined_df = pd.concat(all_data, axis=0, ignore_index=True)

print(combined_df)

In [ ]:
combined_df.columns
combined_df = combined_df[['ID_REF'] + [col for col in combined_df.columns if col != 'ID_REF' and col != 'cell_type'] + ['cell_type']]
combined_df.drop(columns=['VALUE_4', 'VALUE_5'], inplace=True)
combined_df

In [ ]:
combined_df['ID_REF'] = pd.to_numeric(combined_df['ID_REF'], errors='coerce')
combined_df = combined_df.merge(gene_metadata, left_on='ID_REF', right_on='ID', how='inner')

In [ ]:
combined_df = combined_df[['ID_REF', 'VALUE_1', 'VALUE_2', 'VALUE_3', 'cell_type', 'Gene symbol']]
combined_df = combined_df[combined_df['Gene symbol'].notnull()]
combined_df

In [ ]:
pivot_df = combined_df.pivot_table(index=['ID_REF', 'Gene symbol'], columns='cell_type', values=['VALUE_1', 'VALUE_2', 'VALUE_3'])
pivot_df.columns = [f'{i}_{j}' for i, j in pivot_df.columns]

pivot_df.reset_index(inplace=True)
pivot_df.columns = [col.replace('HSC_LT', 'HSC.LT').replace('HSC_ST', 'HSC.ST') for col in pivot_df.columns]
pivot_df.columns = [col.replace('MPP_4', 'MPP.4').replace('MPP_2', 'MPP.2').replace('MPP_3', 'MPP.3') for col in pivot_df.columns]
pivot_df

#protein coding genes only

In [ ]:
X = pivot_df.drop(columns=['ID_REF', 'Gene symbol']).values.T

adata_p = sc.AnnData(X=X)

adata_p.var['gene_id'] = pivot_df['ID_REF'].values
adata_p.var['gene_symbol'] = pivot_df['Gene symbol'].values
adata_p.var.set_index('gene_symbol', inplace=True)

cell_types = [col.split('_')[2] for col in pivot_df.columns[2:]]
replicates = [col.split('_')[1] for col in pivot_df.columns[2:]]

adata_p.obs['cell_type'] = cell_types
adata_p.obs['replicate'] = replicates

adata_p.obs['cell_type'] = adata_p.obs['cell_type'].astype('category')

adata_p

In [ ]:
adata_p.obs['cell_type']

In [ ]:
sc.tl.rank_genes_groups(adata_p, 'cell_type', method='wilcoxon')

#filter to log2FC > 0 and p_val_adj < 0.05

In [ ]:
result = adata_p.uns['rank_genes_groups']
groups = result['names'].dtype.names

for group in groups:
    print(f"Number of genes in group {group}: {len(result['names'][group])}")
    print(f'Ranking genes for group {group}')
    print(result['names'][group])
    print(result['pvals'][group])

In [ ]:
sc.tl.pca(adata_p, n_comps=20)

In [ ]:
pca_var_ratio = adata_p.uns['pca']['variance_ratio']

pc1_var = pca_var_ratio[0] * 100
pc2_var = pca_var_ratio[1] * 100

adata_p.obsm['X_pca'] = adata_p.obsm['X_pca'][:, :2] 

plt.figure(figsize=(10, 7))
sc.pl.pca(adata_p, color='cell_type', size=100, components='1,2', show=False)

plt.xlabel(f'PC1 ({pc1_var:.2f}%)')
plt.ylabel(f'PC2 ({pc2_var:.2f}%)')
plt.xticks(np.arange(-200, 100, 50))
plt.yticks(np.arange(-100, 100, 25))
plt.title('PCA of all reference data')
plt.show()

In [ ]:
# top 100 DE genes for each cell type
top_genes = []
for group in groups:
    top_genes.extend(result['names'][group][:100])
top_genes = list(set(top_genes))

In [ ]:
adata_top = adata_p[:, adata_p.var.index.isin(top_genes)].copy()
sc.tl.pca(adata_top, n_comps=20)

pca_var_ratio = adata_top.uns['pca']['variance_ratio']

pc1_var = pca_var_ratio[0] * 100
pc2_var = pca_var_ratio[1] * 100

plt.figure(figsize=(10, 7))
sc.pl.pca(adata_top, color='cell_type', size=200, components='1,2', show=False)
plt.xlabel(f'PC1 ({pc1_var:.2f}%)')
plt.ylabel(f'PC2 ({pc2_var:.2f}%)')
plt.xticks(np.arange(-40, 40, 20))
plt.yticks(np.arange(-40, 40, 20))
plt.title('PCA of top 100 DE genes')
plt.show()

In [ ]:
names = adata_p.uns['rank_genes_groups']['names']
logfoldchanges = adata_p.uns['rank_genes_groups']['logfoldchanges']

top_genes = {}

for group in names.dtype.names:
    valid_indices = logfoldchanges[group] > 0
    valid_names = names[group][valid_indices]
    
    top_genes[group] = valid_names[:100]


markers = pd.DataFrame(columns=['gene_symbol', 'cell_type'])

for cell_type, genes in top_genes.items():
    temp_df = pd.DataFrame({'gene_symbol': genes, 'cell_type': cell_type})
    markers = pd.concat([markers, temp_df], ignore_index=True)

markers = markers.dropna(subset=['gene_symbol'])
markers.drop_duplicates(inplace=True)
markers

In [ ]:
#filtering to log2fc > 0 and doing differential expression analysis, then ora
ora_estimate = pd.DataFrame()

for i in range(20):
    degs_i = sc.get.rank_genes_groups_df(rna, group=str(i), log2fc_min=0, key='rank_genes_groups_8')
    degs_i.set_index('names', inplace=True)

    ora_i = dc.get_ora_df(degs_i, net=markers, source='cell_type', target='gene_symbol', verbose=True)
    ora_i['group'] = i
    ora_estimate = pd.concat([ora_estimate, ora_i], ignore_index=True)

ora_estimate

In [ ]:
ora_estimate_copy = ora_estimate.copy()
ora_estimate_copy

In [ ]:
order = ['HSC.LT', 'HSC.ST', 'MPP.2', 'MPP.3', 'MPP.4', 'GMP', 'CMP', 'PreGr', 'Gr']
ora_estimate_copy['Term'] = pd.Categorical(ora_estimate_copy['Term'], categories=order, ordered=True)
group_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
ora_estimate_copy['group'] = pd.Categorical(ora_estimate_copy['group'], categories=group_order, ordered=True)
ora_estimate_copy

In [ ]:
# Create the bubble plot
plt.figure(figsize=(12, 8))
bubble_plot = sns.scatterplot(
    data=ora_estimate_copy,
    x='Term',
    y='group',
    size='Overlap ratio',
    hue='FDR p-value',
    hue_norm=(0, 1),
    palette='RdBu_r',
    sizes=(50, 200),
    legend='brief'
)

# Customize plot labels and legend
plt.xlabel('Cell Types')
plt.ylabel('Clusters')
plt.xticks(rotation=90)
bubble_plot.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.yticks(ticks=range(len(group_order)), labels=group_order)
plt.show()

In [ ]:
n_ctypes = 3
# pval and statistic
ora_estimate_copy.sort_values('Combined score', ascending=False, inplace=True)
ctypes_dict = ora_estimate_copy.groupby('group').head(n_ctypes).groupby('group')['Term'].apply(lambda x: list(x)).to_dict()
ctypes_dict



annotations_df['pietras_FC_0_no_fdr'] = ctypes_dict.values()
annotations_df

In [ ]:
ora_estimate_copy = ora_estimate_copy[ora_estimate_copy['FDR p-value'] <= 0.05]
ora_estimate_copy.sort_values('Combined score', ascending=False, inplace=True)
ctypes_dict = ora_estimate_copy.groupby('group').head(n_ctypes).groupby('group')['Term'].apply(lambda x: list(x)).to_dict()
ctypes_dict


annotations_df['pietras_FC_0_fdr_0.05'] = ctypes_dict.values()
annotations_df

## Calculating Gene expression signatures

In [ ]:
plt.figure(figsize=(30, 20))
sc.pl.pca(rna, color='leiden_8', size=10, components='1,2', title='PCA of Clusters with ALL genes', show=False)
pca_var_ratio = rna.uns['pca']['variance_ratio']
pc1_var = pca_var_ratio[0] * 100
pc2_var = pca_var_ratio[1] * 100
plt.xlabel(f'PC1 ({pc1_var:.2f}%)')
plt.ylabel(f'PC2 ({pc2_var:.2f}%)')
plt.xticks(np.arange(-10, 20, 5))
plt.yticks(np.arange(-10, 20, 5))
plt.title('PCA of Clusters with ALL genes')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sc.pl.umap(rna, color='leiden_8', size=1, title='UMAP of Clusters (Leiden)', show=False)
plt.title('UMAP of Clusters with ALL genes')
plt.show()

In [ ]:
adata= rna.copy()
adata

In [ ]:
print(adata.uns['rank_genes_groups_8'].keys())

In [ ]:
## code: compare the differentially expressed genes in our scRNA data to the genes identified for each cell-type, count number of genes in common for each cell type to populate counts_df

# Access the differential expression results
rank_genes_groups_8 = adata.uns['rank_genes_groups_8']  # this is our scRNA data
result = adata_p.uns['rank_genes_groups']  # this is the reference - groups are cell types with differential genes
groups = result['names'].dtype.names

clusters = range(len(adata.obs['leiden_8'].unique()))
cell_types = list(set(groups))

counts_df_1000_up = pd.DataFrame(0, index=clusters, columns=cell_types)
counts_df_1000_down = pd.DataFrame(0, index=clusters, columns=cell_types)

reference_genes_up = {}
reference_genes_down = {}

for cell_type in groups:
    print(f"Processing cell type: {cell_type}")

    reference_genes = result['names'][cell_type]
    reference_logfoldchanges = result['logfoldchanges'][cell_type]
    
    reference_genes_up[cell_type] = [reference_genes[i] for i in range(len(reference_genes)) if reference_logfoldchanges[i] > 0][:100]
    reference_genes_down[cell_type] = [reference_genes[i] for i in range(len(reference_genes)) if reference_logfoldchanges[i] < 0][:100]

    print(f"Top upregulated genes for cell type {cell_type}: {reference_genes_up[cell_type]}")
    print(f"Top downregulated genes for cell type {cell_type}: {reference_genes_down[cell_type]}")


for cluster in clusters:
    print(f"Processing cluster: {cluster}")
    
    genes = rank_genes_groups_8['names'][str(cluster)]
    logfoldchanges = rank_genes_groups_8['logfoldchanges'][str(cluster)]
    pvals_adj = rank_genes_groups_8['pvals_adj'][str(cluster)]
    
    # Separate the significant upregulated and downregulated genes
    significant_genes_up = [genes[i] for i in range(len(genes)) if logfoldchanges[i] > 0][:1000]
    significant_genes_down = [genes[i] for i in range(len(genes)) if logfoldchanges[i] < 0][:1000]
    
    print(f"Top upregulated genes for cluster {cluster}: {significant_genes_up}")
    print(f"Top downregulated genes for cluster {cluster}: {significant_genes_down}")
    
    for gene in significant_genes_up:
        for cell_type in groups:
            if gene in reference_genes_up[cell_type]:
                counts_df_1000_up.at[cluster, cell_type] += 1

    for gene in significant_genes_down:
        for cell_type in groups:
            if gene in reference_genes_down[cell_type]:
                counts_df_1000_down.at[cluster, cell_type] += 1



In [ ]:
counts_df_1000_up['cluster'] = counts_df_1000_up.index
counts_df_1000_down['cluster'] = counts_df_1000_down.index

melted_df_up = counts_df_1000_up.melt(id_vars='cluster', var_name='cell_type', value_name='count_up')
melted_df_down = counts_df_1000_down.melt(id_vars='cluster', var_name='cell_type', value_name='count_down')

melted_df_up = melted_df_up[melted_df_up['count_up'] > 0]
melted_df_down = melted_df_down[melted_df_down['count_down'] > 0]

order = ['HSC.LT', 'HSC.ST', 'MPP.2', 'MPP.3', 'MPP.4', 'GMP', 'CMP', 'PreGr', 'Gr']
melted_df_up['cell_type'] = pd.Categorical(melted_df_up['cell_type'], categories=order, ordered=True)
group_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
melted_df_up['cluster'] = pd.Categorical(melted_df_up['cluster'], categories=group_order, ordered=True)
melted_df_up

order = ['HSC.LT', 'HSC.ST', 'MPP.2', 'MPP.3', 'MPP.4', 'GMP', 'CMP', 'PreGr', 'Gr']
melted_df_down['cell_type'] = pd.Categorical(melted_df_down['cell_type'], categories=order, ordered=True)
group_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
melted_df_down['cluster'] = pd.Categorical(melted_df_down['cluster'], categories=group_order, ordered=True)
melted_df_down

plt.figure(figsize=(12, 10))

sns.scatterplot(
    data=melted_df_up,
    x='cell_type',
    y='cluster',
    size='count_up',
    hue='count_up',
    palette='Reds',
    sizes=(20, 200),
    legend='brief',
    marker='o'
)

sns.scatterplot(
    data=melted_df_down,
    x='cell_type',
    y='cluster',
    size='count_down',
    hue='count_down',
    palette='Blues',
    sizes=(20, 200),
    legend='brief',
    marker='X'
)

plt.yticks(range(0, 20))

plt.xlabel('Cell Type')
plt.ylabel('Cluster')
plt.title('Top 1000 genes for each cluster (Upregulated and Downregulated)')
plt.xticks(rotation=90)

up_handles, up_labels = plt.gca().get_legend_handles_labels()
plt.legend(up_handles, up_labels, bbox_to_anchor=(1.05, 1), loc='upper left')

down_handles, down_labels = plt.gca().get_legend_handles_labels()
plt.legend(down_handles, down_labels, bbox_to_anchor=(1.05, 0.5), loc='upper left')

plt.show()

In [ ]:
rank_genes_groups_8 = adata.uns['rank_genes_groups_8']  # this is our scRNA data
result = adata_p.uns['rank_genes_groups']  # this is the reference - groups are cell types with differential genes
groups = result['names'].dtype.names

clusters = range(len(adata.obs['leiden_8'].unique()))
cell_types = list(set(groups))

counts_df_2000_up = pd.DataFrame(0, index=clusters, columns=cell_types)
counts_df_2000_down = pd.DataFrame(0, index=clusters, columns=cell_types)

reference_genes_up = {}
reference_genes_down = {}

for cell_type in groups:
    print(f"Processing cell type: {cell_type}")

    reference_genes = result['names'][cell_type]
    reference_logfoldchanges = result['logfoldchanges'][cell_type]
    
    reference_genes_up[cell_type] = [reference_genes[i] for i in range(len(reference_genes)) if reference_logfoldchanges[i] > 0][:100]
    reference_genes_down[cell_type] = [reference_genes[i] for i in range(len(reference_genes)) if reference_logfoldchanges[i] < 0][:100]

    print(f"Top upregulated genes for cell type {cell_type}: {reference_genes_up[cell_type]}")
    print(f"Top downregulated genes for cell type {cell_type}: {reference_genes_down[cell_type]}")


for cluster in clusters:
    print(f"Processing cluster: {cluster}")
    
    genes = rank_genes_groups_8['names'][str(cluster)]
    logfoldchanges = rank_genes_groups_8['logfoldchanges'][str(cluster)]
    pvals_adj = rank_genes_groups_8['pvals_adj'][str(cluster)]
    
    # Separate the significant upregulated and downregulated genes
    significant_genes_up = [genes[i] for i in range(len(genes)) if logfoldchanges[i] > 0][:2000]
    significant_genes_down = [genes[i] for i in range(len(genes)) if logfoldchanges[i] < 0][:2000]
    
    print(f"Top upregulated genes for cluster {cluster}: {significant_genes_up}")
    print(f"Top downregulated genes for cluster {cluster}: {significant_genes_down}")
    
    for gene in significant_genes_up:
        for cell_type in groups:
            if gene in reference_genes_up[cell_type]:
                counts_df_2000_up.at[cluster, cell_type] += 1

    for gene in significant_genes_down:
        for cell_type in groups:
            if gene in reference_genes_down[cell_type]:
                counts_df_2000_down.at[cluster, cell_type] += 1


In [ ]:
counts_df_2000_up['cluster'] = counts_df_2000_up.index
counts_df_2000_down['cluster'] = counts_df_2000_down.index

melted_df_up = counts_df_2000_up.melt(id_vars='cluster', var_name='cell_type', value_name='count_up')
melted_df_down = counts_df_2000_down.melt(id_vars='cluster', var_name='cell_type', value_name='count_down')

melted_df_up = melted_df_up[melted_df_up['count_up'] > 0]
melted_df_down = melted_df_down[melted_df_down['count_down'] > 0]

order = ['HSC.LT', 'HSC.ST', 'MPP.2', 'MPP.3', 'MPP.4', 'GMP', 'CMP', 'PreGr', 'Gr']
melted_df_up['cell_type'] = pd.Categorical(melted_df_up['cell_type'], categories=order, ordered=True)
group_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
melted_df_up['cluster'] = pd.Categorical(melted_df_up['cluster'], categories=group_order, ordered=True)
melted_df_up

order = ['HSC.LT', 'HSC.ST', 'MPP.2', 'MPP.3', 'MPP.4', 'GMP', 'CMP', 'PreGr', 'Gr']
melted_df_down['cell_type'] = pd.Categorical(melted_df_down['cell_type'], categories=order, ordered=True)
group_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
melted_df_down['cluster'] = pd.Categorical(melted_df_down['cluster'], categories=group_order, ordered=True)
melted_df_down

plt.figure(figsize=(12, 10))

sns.scatterplot(
    data=melted_df_up,
    x='cell_type',
    y='cluster',
    size='count_up',
    hue='count_up',
    palette='Reds',
    sizes=(20, 200),
    legend='brief',
    marker='o'
)

sns.scatterplot(
    data=melted_df_down,
    x='cell_type',
    y='cluster',
    size='count_down',
    hue='count_down',
    palette='Blues',
    sizes=(20, 200),
    legend='brief',
    marker='X'
)

plt.yticks(range(0, 20))

plt.xlabel('Cell Type')
plt.ylabel('Cluster')
plt.title('Top 2000 genes for each cluster (Upregulated and Downregulated)')
plt.xticks(rotation=90)

up_handles, up_labels = plt.gca().get_legend_handles_labels()
plt.legend(up_handles, up_labels, bbox_to_anchor=(1.05, 1), loc='upper left')

down_handles, down_labels = plt.gca().get_legend_handles_labels()
plt.legend(down_handles, down_labels, bbox_to_anchor=(1.05, 0.5), loc='upper left')

plt.show()


## Annotation + Visualize in UMAP

In [ ]:
annotations_df

In [ ]:
# annotation_dict = {'0': 'HSC', 
#     '1': 'MPP4', 
#     '2': 'HSC', 
#     '3': 'MPP3', 
#     '4': 'MPP3', 
#     '5': 'MkP', 
#     '6': 'MPP2', 
#     '7': 'GMP', 
#     '8': 'cMoP', 
#     '9': 'GP', 
#     '10': 'GP', 
#     '11': 'GP', 
#     '12': 'Pre-CFU-E', 
#     '13': 'Pre-CFU-E', 
#     '14': 'CFU-E', 
#     '15': 'cMoP', 
#     '16': 'GP',
#     '17': 'GP',
#     '18': 'Pre-CFU-E'
# }

annotation_dict = {'0': 'LT-HSC', 
    '1': 'ST-HSC',  
    '2': 'MPP4', 
    '3': 'MkP', 
    '4': 'MPP3', 
    '5': 'LT-HSC', 
    '6': 'MPP3', 
    '7': 'GMP', 
    '8': 'MPP2', 
    '9': 'GP', 
    '10': 'GP', 
    '11': 'Pre-CFU-E', 
    '12': 'CFU-E', 
    '13': 'GP', 
    '14': 'cMoP', 
    '15': 'MkP', 
    '16': 'BaP',
    '17': 'GP',
    '18': 'MPP4',
    '19': 'Pre-CFU-E'
}


annotation_dict

In [ ]:
rna_copy = rna.copy()

In [ ]:
rna_copy.obs['leiden_8']

In [ ]:
order = ['LT-HSC', 'ST-HSC', 'MPP2', 'MPP3', 'MPP4', 'GMP', 'cMoP', 'GP', 'MkP', 'BaP','Pre-CFU-E', 'CFU-E']

rna_copy.obs['cell_type'] = rna_copy.obs['leiden_8'].map(annotation_dict)

rna_copy.obs['cell_type'] = pd.Categorical(rna_copy.obs['cell_type'], categories=order, ordered=True)

In [ ]:
# rna_copy.write('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/scRNA_res0.8_annotated_rna_qc_added_strict.h5ad')
rna_copy = sc.read_h5ad('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/adata_objects/scRNA_res0.8_annotated_rna_qc_added_strict.h5ad')

In [ ]:
cell_data = pd.DataFrame({
    'cell_barcode': rna_copy.obs.index,
    'cell_batch': rna_copy.obs['batch'],
    'cell_type': rna_copy.obs['cell_type']
})

map = {'Pool140_1': '0', 'Pool140_2': '1', 'Pool140_3': '2', 'Pool140_4': '3', 'Pool140_5': '4', 'Pool140_6': '5', 'Pool140_7': '6', 'Pool140_8': '7', 'Pool140_9': '8', 'Pool140_10': '9', 'Pool140_11': '10', 'Pool140_12': '11', 'Pool140_13': '12'}


map_swapped = {v: k for k, v in map.items()}

cell_data['Pool'] = cell_data['cell_batch'].map(map_swapped)

batch_to_genotype2 = dict(zip(meta['batch'], meta['Genotype2']))

# Map 'Genotype2' to cell_data based on 'cell_batch' using the batch_to_genotype2 dictionary
cell_data['Genotype2'] = cell_data['cell_batch'].map(batch_to_genotype2)

cell_data.to_csv('/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/Pool_140_cell_barcodes_w_celltype_annotation.csv', index=False, header=True)

In [ ]:
sc.pl.umap(rna_copy, color='cell_type', legend_loc='right margin', palette='tab20', show=False)

plt.gca().tick_params(axis='both', which='both', labelsize=10)
plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.gca().yaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('scRNA-seq')

plt.grid(False)
plt.xticks([-5,0,5,10,15])

handles, labels = plt.gca().get_legend_handles_labels()
colors = [handle.get_facecolor()[0] for handle in handles]
print(colors)
plt.savefig("/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/figures/UMAP_qc_added_strict.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
rna_copy.obs['batch'].value_counts()

In [ ]:
gmt_file = '/home/beb528/ref_annots/msigdb/msigdb_v2023.2.Mm_GMTs/mh.all.v2023.2.Mm.symbols.gmt'
select_terms = [
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_E2F_TARGETS'
]

def parse_gmt_to_list(file, terms):
    gene_list = []
    with open(file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            term_name = parts[0]
            if term_name in terms:
                genes = parts[2:] 
                gene_list.extend(genes)
    return list(set(gene_list))

gsea_genes = parse_gmt_to_list(gmt_file, select_terms)
gsea_genes = [gene for gene in gsea_genes if gene in rna_copy.var_names]

In [ ]:
# Inflammatory Pathways
sc.pl.heatmap(
    rna_copy,
    var_names=gsea_genes,   
    groupby='cell_type',         
    layer='counts',              
    cmap='coolwarm',              
    standard_scale='obs'      
)

### Genotype analysis: After annotation

In [ ]:
rna_copy.obs['cell_type']

In [ ]:
meta = pd.read_csv('/home/rav589/ben_scratch/scrnaseq/tet2_stag2_mice_Pool140/samples.csv', sep=',')
meta

map = {'Pool140_1': '0', 'Pool140_2': '1', 'Pool140_3': '2', 'Pool140_4': '3', 'Pool140_5': '4', 'Pool140_6': '5', 'Pool140_7': '6', 'Pool140_8': '7', 'Pool140_9': '8', 'Pool140_10': '9', 'Pool140_11': '10', 'Pool140_12': '11', 'Pool140_13': '12'}

meta['batch'] = meta['Pool'].map(map)
meta['Genotype'] = meta['Genotype'].str.replace('\d+', '', regex=True)
meta['Genotype'] = meta['Genotype'].str.replace('STAG', 'Stag2', regex=True)
meta

In [ ]:
map = meta.set_index('batch')['Genotype'].to_dict()
rna_copy.obs['genotype'] = rna_copy.obs['batch'].map(map)

order = ['WT', 'CHIP', 'MDS', 'Stag2']
rna_copy.obs['genotype'] = pd.Categorical(rna_copy.obs['genotype'], categories=order, ordered=True)
rna_copy.obs['genotype']

In [ ]:
map = meta.set_index('batch')['Genotype2'].to_dict()
rna_copy.obs['genotype2'] = rna_copy.obs['batch'].map(map)

order = ['WT1','WT2','WT3','WT4', 'CHIP1','CHIP2','CHIP3', 'MDS1','MDS2', 'MDS3', 'STAG1','STAG2','STAG3']
rna_copy.obs['genotype2'] = pd.Categorical(rna_copy.obs['genotype2'], categories=order, ordered=True)
rna_copy.obs['genotype2']

In [ ]:
genotype = rna_copy.obs['genotype2']
cell_type = rna_copy.obs['cell_type']

df = pd.DataFrame({'genotype': genotype, 'cell_type': cell_type})

crosstab = pd.crosstab(df['genotype'], df['cell_type'])
order = ['LT-HSC', 'ST-HSC', 'MPP2', 'MPP3', 'MPP4', 'GMP', 'cMoP', 'GP', 'MkP', 'BaP','Pre-CFU-E', 'CFU-E']
crosstab = crosstab[order]

crosstab_flipped = crosstab[order[::-1]]

ax = crosstab_flipped.plot(kind='bar', stacked=True, figsize=(8, 6), color=colors[::-1], width=0.85)
plt.xlabel('Genotype')
plt.ylabel('Cell Counts')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(False)

plt.savefig("/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/rawcounts_barplot_qc_added_strict.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
crosstab = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

crosstab_flipped = crosstab[order[::-1]]

ax = crosstab_flipped.plot(kind='bar', stacked=True, figsize=(10, 7), color=colors[::-1], width = 0.85)
plt.xlabel('Genotype')
plt.ylabel('% of total')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(False)

plt.savefig("/home/rav589/scratch/SC-RNASEQ/tet2_stag2_pool140/percentage_barplot_qc_added_strict.pdf", format="pdf", bbox_inches="tight")

plt.show()